# GPU03 — خ۸ (F08): سلسله‌مراتبی و بیزی — NUTS روی GPU

> بند 7.17 `doc/WBS-phase7-modeling.md` · اسپرینت C، ردیف «خ۸ بیزی سلسله‌مراتبی».

**چرا این خانواده منطبق‌ترین با ساختار مسئله است:** F10 (ICC روز=۰.۲۲۵ و
سلف=۰.۲۰۵ ⇒ اثرات تصادفی **متقاطع**)، F59 (۸۳٪ واریانس = شوک مشترک روز)، F07
(بیش‌پراکندگی صعودی ⇒ Beta-Binomial نه دوجمله‌ای).

⭐ **و یک مزیت که هیچ خانواده‌ی دیگری ندارد:** روز آزمون یک روز **جدید** است، پس
اثر تصادفی‌اش معلوم نیست و باید از پیشین پسین‌آموخته قرعه بخورد. مدل‌های نقطه‌ای
عملاً وانمود می‌کنند «شوک روز آینده صفر است» — یعنی همان چیزی که ۸۳٪ واریانس را
می‌سازد از عدم‌قطعیت حذف می‌شود. بند 7.17 انتظار دارد این خانواده حتی اگر pinball
را نبرد، **کالیبراسیون** بهتری بدهد — و بند ۶.۴ کالیبراسیون را معیار اصلی می‌داند.

| model_id | عضو WBS | ساختار |
|---|---|---|
| `bhm_beta_binomial_restaurant` | ۱/۵ | فقط اثر تصادفی سلف |
| `bhm_beta_binomial_crossed` | ⭐ ۲/۵ | روز + سلف متقاطع |
| `bhm_varying_dispersion` | ⭐ ۶ | $\phi = f(\log Res)$ — پاسخ به F06/F07 |

**بودجه‌ی هدف: ~۹۰ دقیقه.**

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [1]:
!pip install -q numpyro arviz optuna mlflow tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.8/394.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [2]:
MODE = "kaggle"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/working/t.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
# with zipfile.ZipFile(bundle) as z:
#     z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))

import os
import shutil

src_dir = "/kaggle/input/datasets/mvajhi/bundle"
dest_dir = "/kaggle/working/phase7"

os.makedirs(dest_dir, exist_ok=True)

# Copy all contents from Kaggle input to working/my_dir
for item in os.listdir(src_dir):
    src_path = os.path.join(src_dir, item)
    dest_path = os.path.join(dest_dir, item)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dest_path)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

محتوای بسته: ['AGENTS.md', 'BUNDLE_INFO.json', 'data', 'doc', 'reports', 'requirements-gpu.txt', 'src']


## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [3]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.gpu_runner import load_l1
data = load_l1()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

✅ دروازه‌ی انصاف A1 پاس شد · cv_folds_hash=bd08d6f7c801… · data_snapshot_hash=68b4cb8517d2…
L1 — 7,579 ردیف، 5 fold (fold0: 3,556→870 · fold1: 4,426→860 · fold2: 5,286→185 · fold3: 5,471→1,025 · fold4: 6,496→940)


## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [4]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [5]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

Sun Aug 16 21:20:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

INFO:2026-08-16 21:21:01,049:jax._src.xla_bridge:822: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-08-16 21:21:01,049 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


{'platform': 'Linux-6.12.90+-x86_64-with-glibc2.35',
 'python': '3.12.13',
 'torch': '2.10.0+cu128',
 'cuda': '12.8',
 'device': 'cuda',
 'gpu_name': 'Tesla T4',
 'gpu_memory_gb': 15.64,
 'jax': '0.7.2',
 'jax_devices': ['cuda:0', 'cuda:1']}

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [6]:
from pathlib import Path
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "kaggle"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

# reports/gpu/ را همین‌جا می‌سازیم — سلول‌های ۷-ب/۷-ج (PPC، حساسیت پیشین) مستقیم
# CSV آن‌جا می‌نویسند، پیش از آنکه save_family_report/package_outputs بسازدش
Path("reports/gpu").mkdir(parents=True, exist_ok=True)


MLflow → /kaggle/working/phase7/mlruns_gpu


## سلول ۷-الف — R0 + چک‌لیست همگرایی بند 7.17.3

برای مدل بیزی، «شاهد همگرایی» قاعده‌ی A6 جایش را به چک‌لیست بند 7.17.3 می‌دهد:
R̂ < ۱.۰۱ · ESS > ۴۰۰ · صفر divergence. `convergence_report` هر سه را عدد می‌دهد،
نه بررسی چشمی نمودار.

In [7]:
import pandas as pd
from src.models.families import f08_bayesian as fam
from src.models.gpu_runner import smoke_test
from src.models.axes import TUNING_TAU

MODEL_IDS = ["bhm_beta_binomial_restaurant", "bhm_beta_binomial_crossed",
             "bhm_varying_dispersion"]

smoke, diagnostics = [], []
tr0, te0 = data.folds[0]
for mid in MODEL_IDS:
    smoke.append(smoke_test(fam.FITTERS[mid], data,
                            hyperparams={"num_warmup": 300, "num_samples": 300}))
    model = fam.FITTERS[mid].fit(tr0, TUNING_TAU, num_warmup=300, num_samples=300)
    diagnostics.append({"مدل": mid, **model.diagnostics})

convergence_table = pd.DataFrame(diagnostics)
convergence_table

R0 bhm_beta_binomial_restaurant pinball=0.01267 (B3=0.01375) پوشش=0.189 R²=-0.276 61.4s
R0 bhm_beta_binomial_crossed    pinball=0.01256 (B3=0.01375) پوشش=0.198 R²=-0.297 79.4s
R0 bhm_varying_dispersion       pinball=0.01291 (B3=0.01375) پوشش=0.170 R²=-0.360 78.8s


,مدل,max_r_hat,min_ess,n_divergences,divergence_rate,passes_rhat,passes_ess,passes_divergence
0,bhm_beta_binomial_restaurant,1.043303,60.489549,0,0.0,False,False,True
1,bhm_beta_binomial_crossed,1.048802,64.406981,0,0.0,False,False,True
2,bhm_varying_dispersion,1.019443,93.703731,0,0.0,False,False,True


## سلول ۷-ب — ⭐ Posterior Predictive Check (بند 7.17.3، مهم‌ترین بند چک‌لیست)

> «مدلی که میانگین را درست می‌زند ولی چولگی را بازتولید نمی‌کند، کوانتایل‌هایش غلط
> است.» هدف: چولگی ~۴.۰۶ (F02) و تورم صفر ~۴.۹٪ (F03).

In [8]:
ppc_rows = []
for mid in MODEL_IDS:
    model = fam.FITTERS[mid].fit(tr0, TUNING_TAU, num_warmup=300, num_samples=300)
    ppc_rows.append({"مدل": mid, **fam.posterior_predictive_check(model, te0)})
ppc_table = pd.DataFrame(ppc_rows)
ppc_table.to_csv("reports/gpu/F08_ppc.csv", index=False)
ppc_table

,مدل,sim_skew,actual_skew,target_skew_F02,sim_zero_share,actual_zero_share,target_zero_share_F03,sim_mean,actual_mean
0,bhm_beta_binomial_restaurant,1.813758,3.05717,4.06,0.056261,0.062069,0.049,0.096954,0.106901
1,bhm_beta_binomial_crossed,1.786370,3.05717,4.06,0.056132,0.062069,0.049,0.097538,0.106901
2,bhm_varying_dispersion,2.108490,3.05717,4.06,0.066316,0.062069,0.049,0.098433,0.106901


## سلول ۷-ج — تحلیل حساسیت پیشین (اجباری، بند 7.17.2)

سه مجموعه پیشین (`tight` / `default` / `wide`) روی fold۰. اگر نتیجه به پیشین حساس
باشد، هر ادعایی درباره‌ی این خانواده باید با آن حساسیت گزارش شود.

In [9]:
from src.baselines import operational_metrics

prior_rows = []
for preset in fam.PRIOR_PRESETS:
    m = fam.FITTERS["bhm_beta_binomial_crossed"].fit(
        tr0, TUNING_TAU, prior_preset=preset, num_warmup=300, num_samples=300)
    pred = m.predict(te0, TUNING_TAU)
    met = operational_metrics(te0, pred, TUNING_TAU)
    prior_rows.append({"پیشین": preset, **fam.PRIOR_PRESETS[preset],
                       "pinball": round(met["pinball"], 5),
                       "پوشش": round(met["coverage"], 4),
                       "max_R_hat": round(m.diagnostics["max_r_hat"], 4),
                       "divergences": m.diagnostics["n_divergences"]})
prior_table = pd.DataFrame(prior_rows)
prior_table.to_csv("reports/gpu/F08_prior_sensitivity.csv", index=False)
prior_table

,پیشین,prior_sigma,prior_beta,pinball,پوشش,max_R_hat,divergences
0,tight,0.5,1.0,0.01251,0.1862,1.0153,0
1,default,1.0,2.5,0.01250,0.1920,1.0218,0
2,wide,2.0,5.0,0.01261,0.1931,1.0188,0


## سلول ۷-د — R2: تنظیم با بودجه‌ی زمانی

⚠️ فضای این خانواده عمداً **کاردینالیتی محدود** (۲۷) اعلام شده — بدون آن، جدول
بودجه ۶۰ trial می‌داد که با NUTS یعنی چند ده ساعت. دقیقاً همان اشتباهی که یافته‌ی ۱۲
مستند کرده است.

In [10]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

BUDGET_MINUTES = {"bhm_beta_binomial_restaurant": 12,
                  "bhm_beta_binomial_crossed": 22,
                  "bhm_varying_dispersion": 22}

studies = [run_gpu_study(fam.FITTERS[mid], SPACES[mid].fn, data,
                         family=fam.FAMILY, feature_set=fam.FEATURE_SET,
                         budget_minutes=BUDGET_MINUTES[mid], compute=COMPUTE, seed=42)
           for mid in MODEL_IDS]

2026/08/16 21:35:23 INFO mlflow.tracking.fluent: Experiment with name 'phase7' does not exist. Creating a new experiment.



R2 — F08/bhm_beta_binomial_restaurant (L1) · τ=0.2 · بودجه=12 دقیقه · دستگاه=cuda · مرجع B3=0.01590


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   0 | pinball=0.01567 🎯 | بهترین=0.01567 | 351.0s | گذشته=  5.9/12 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

...

  trial   0 | pinball=0.01575 🎯 | بهترین=0.01575 | 457.4s | گذشته=  7.6/22 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   1 | pinball=0.01570 🎯 | بهترین=0.01570 | 642.7s | گذشته= 18.3/22 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   2 | pinball=0.01477 🎯 | بهترین=0.01477 | 608.6s | گذشته= 28.5/22 دقیقه
⏱️  بودجه‌ی زمانی (22 دقیقه) تمام شد پس از 3 trial — با بهترین نتیجه‌ی تا این لحظه ادامه می‌دهیم.

✅ bhm_varying_dispersion: بهترین pinball=0.01477 در برابر B3=0.01590 (برد) · 3 trial · همگرا(A6)=✅ · پایداری=4/5


## سلول ۷-ه — قهرمان + ACI + DM

⭐ نکته‌ی مورد انتظار: ستون «پوشش» این خانواده. بند 7.17.5 می‌گوید مقایسه‌ی پوشش
کوانتایل پسین با مدل‌های غیربیزی «جایی است که این خانواده احتمالاً می‌برد» — حتی
اگر pinball را نبرد.

In [11]:
from src.models.gpu_runner import finalize_champion

best = min(studies, key=lambda s: s.best_pinball)
print(f"قهرمان: {best.model_id} (pinball={best.best_pinball:.5f})\n")
champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234),
                               compute=COMPUTE, run_aci=True)]

قهرمان: bhm_varying_dispersion (pinball=0.01477)

  seed 42: pinball(ردیفی)=0.01235
  seed 1234: pinball(ردیفی)=0.01221
  ACI: پوشش=0.2034 (شکاف +0.0034) · pinball=0.01252

🏁 bhm_varying_dispersion: pinball(ردیفی)=0.01221 در برابر B3=0.01335 · DM p=0.0000 ✅ معنادار · 30 فایل مدل ذخیره شد


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

## سلول ۷-و — راستی‌آزمایی مدل ذخیره‌شده

مدل بیزی وزن ندارد، **توزیع پسین** دارد — همان چیزی که در `.npz` ذخیره شده. با
همین فایل می‌شود بدون هیچ نمونه‌گیری دوباره، برای هر داده‌ی جدید و **هر τ** پیش‌بینی
گرفت. سلول زیر همین را نشان می‌دهد.

In [12]:
import numpy as np
from pathlib import Path
from src.models.axes import TAU_GRID

stem = Path(f"models/gpu/F08/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
print("τ | پیش‌بینی میانگین (از همان پسین ذخیره‌شده، بدون نمونه‌گیری دوباره)")
for t in TAU_GRID:
    print(f"{t:.2f} | {reloaded.predict(data.folds[0][1], t).mean():.5f}")
print("\nتشخیص همگرایی ذخیره‌شده:", reloaded.diagnostics)
print("✅ مدل ذخیره‌شده قابل استفاده است")

τ | پیش‌بینی میانگین (از همان پسین ذخیره‌شده، بدون نمونه‌گیری دوباره)
0.02 | 0.02154
0.05 | 0.02959
0.10 | 0.03832
0.15 | 0.04555
0.20 | 0.05215

تشخیص همگرایی ذخیره‌شده: {'max_r_hat': 1.005652904510498, 'min_ess': 193.15357232449782, 'n_divergences': 0, 'divergence_rate': 0.0, 'passes_rhat': True, 'passes_ess': False, 'passes_divergence': True}
✅ مدل ذخیره‌شده قابل استفاده است


## سلول ۷-ز — گزارش فارسی کامل

In [13]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    "چک‌لیست همگرایی بند 7.17.3 جای قاعده‌ی A6 را می‌گیرد — جدول `F08_ppc.csv` و ستون‌های R̂/ESS.",
    "اثر روزِ آزمون از پیشین قرعه می‌خورد نه از پسین — این تفاوت کالیبراسیون بیزی با مدل نقطه‌ای است.",
    "تحلیل حساسیت پیشین اجباری اجرا شد: `reports/gpu/F08_prior_sensitivity.csv`.",
    "پارامترسازی غیرمرکزی (z~N(0,1); u=σ·z) — بند 7.17.2، بدون آن NUTS در قیف نیل واگرا می‌شود.",
]
report = render_family_report("F08", "خ۸ — سلسله‌مراتبی و بیزی (NUTS روی GPU)",
                              studies, champions, smoke, DEVICE, notes)
report += ("\n## چک‌لیست همگرایی (بند 7.17.3)\n\n"
           + convergence_table.to_markdown(index=False)
           + "\n\n## Posterior Predictive Check\n\n" + ppc_table.to_markdown(index=False)
           + "\n\n## حساسیت پیشین\n\n" + prior_table.to_markdown(index=False) + "\n")
save_family_report("F08", report, "F08_bayesian_L1")
print(report)

گزارش نوشته شد: /kaggle/working/phase7/reports/gpu/F08_bayesian_L1.md
# خ۸ — سلسله‌مراتبی و بیزی (NUTS روی GPU)

> اجرای GPU، بند 7.8 `doc/WBS-phase7-modeling.md`. خانواده F08. سخت‌افزار: Tesla T4 · torch=2.10.0+cu128 · CUDA=12.8

## R0 — آزمایش دود (اجراپذیری + سیم‌چین نشتی)

| مدل | pinball | B3 | پوشش | R² | زمان |
|---|---|---|---|---|---|
| `bhm_beta_binomial_restaurant` | 0.01267 | 0.01375 | 0.189 | -0.276 | 61.4s |
| `bhm_beta_binomial_crossed` | 0.01256 | 0.01375 | 0.198 | -0.297 | 79.4s |
| `bhm_varying_dispersion` | 0.01291 | 0.01375 | 0.170 | -0.360 | 78.8s |

## R2 — تنظیم با بودجه‌ی زمانی

| مدل | بهترین pinball | B3 | trial | همگرا (A6) | پایداری (۷.۶.۳) | شکست | ساعت-هسته |
|---|---|---|---|---|---|---|---|
| `bhm_varying_dispersion` 🎯 | **0.01477** | 0.01590 | 3 | ✅ | 4/5 | 0 | 0.47h |
| `bhm_beta_binomial_crossed` 🎯 | **0.01480** | 0.01590 | 3 | ✅ | 2/5 | 0 | 0.49h |
| `bhm_beta_binomial_restaurant` 🎯 | **0.01559** | 0.01590 | 2 | ✅ | 4/5 | 0 | 0.24h |

## S3 — قهرمان‌

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F08_bayesian_L1.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [14]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F08_bayesian_L1", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند


📦 بسته‌بندی شد: 653 فایل (13.6 MB خام) → 1 تکه در gpu_outputs/
   gpu_outputs_F08_bayesian_L1.zip  13.3 MB

بازیابی محلی:
   unzip gpu_outputs_F08_bayesian_L1.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```